<a href="https://colab.research.google.com/github/jamesemcnally/critical-listener/blob/main/recommender%2Bexplainer_in_action.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install anthropic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.7/960.7 kB 31.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import anthropic
import re
from google.colab import drive, userdata

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

# Load recommendation database and review text
all_recs_df = pd.read_parquet(f"{DRIVE_DIR}/recommendations_album_level_avg_embeddings.parquet")
df = pd.read_parquet(f"{DRIVE_DIR}/merged_dataset_masked.parquet")

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

print(f"Loaded {len(all_recs_df):,} recommendation rows")
print(f"Loaded {len(df):,} reviews")

# --- Explainer ---
def explain_pair_simple(query_artist, query_album, rec_artist, rec_album, n=3):
    q_mask = (
        (df["album_norm"] == query_album.lower().strip()) &
        (df["artist_norm"] == query_artist.lower().strip())
    )
    r_mask = (
        (df["album_norm"] == rec_album.lower().strip()) &
        (df["artist_norm"] == rec_artist.lower().strip())
    )

    q_reviews = df[q_mask]
    r_reviews = df[r_mask]

    if q_reviews.empty or r_reviews.empty:
        print(f"  Review not found for one or both albums — skipping explainer")
        return

    q_text   = q_reviews.iloc[0]["cleaned_text"]
    r_text   = r_reviews.iloc[0]["cleaned_text"]
    q_source = q_reviews.iloc[0]["dataset"]
    r_source = r_reviews.iloc[0]["dataset"]
    q_label  = f"{query_artist} — {query_album} ({q_source})"
    r_label  = f"{rec_artist} — {rec_album} ({r_source})"

    prompt = f"""You are analyzing two music reviews to find their most meaningful shared qualities.

Identify up to {n} significant qualities that both albums genuinely share, based ONLY on what is actually written in the reviews.
If fewer than {n} genuine connections exist, list only those that are real — do not pad with weak or generic observations.
If there are no meaningful connections, say so plainly.

For each shared quality:
- State it in one clear phrase
- Quote the specific language from each review that supports it (verbatim, in quotation marks, attributed to Review A or Review B)

Album A: {q_label}
Review A:
{q_text[:2000]}

Album B: {r_label}
Review B:
{r_text[:2000]}

Return a numbered list only. No introduction, no conclusion."""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )

    print(f"\n{'='*60}")
    print(f"  {q_label}")
    print(f"  → {r_label}")
    print(f"{'='*60}")
    print(response.content[0].text.strip())

# --- Main function ---
def recommend_and_explain(query_artist, query_album, top_k=3):
    artist_norm = query_artist.lower().strip()
    album_norm  = query_album.lower().strip()

    recs = all_recs_df[
        (all_recs_df["query_artist"] == artist_norm) &
        (all_recs_df["query_album"]  == album_norm)
    ].sort_values("rank").head(top_k)

    if recs.empty:
        print(f"Not found in recommendation database: {query_artist} — {query_album}")
        return

    print(f"\nTop {top_k} recommendations for: {query_artist} — {query_album}\n")
    for _, row in recs.iterrows():
        print(f"  #{int(row['rank'])}: {row['rec_artist']} — {row['rec_album']} (score: {row['score']})")

    print("\nRunning explainer on top recommendations...")
    for _, row in recs.iterrows():
        explain_pair_simple(
            query_artist, album_norm,
            row["rec_artist"], row["rec_album"]
        )

Mounted at /content/drive
Loaded 447,160 recommendation rows
Loaded 48,189 reviews


In [5]:
albums_to_demo = [
    ("radiohead", "kid a"),
    ("joni mitchell", "blue"),
    ("charli xcx", "pop 2"),
    ("janelle monáe", "the archandroid"),
    ("frank ocean", "channel orange"),
    ("taylor swift", "folklore"),
    ("kendrick lamar", "to pimp a butterfly"),
    ("sufjan stevens", "illinois"),
    ("bon iver", "for emma, forever ago"),
    ("beyoncé", "lemonade"),
    ("outkast", "speakerboxxx/the love below"),
]

for artist, album in albums_to_demo:
    print(f"\n{'#'*70}")
    print(f"  SEED: {artist.title()} — {album.title()}")
    print(f"{'#'*70}")
    recommend_and_explain(artist, album, top_k=3)


######################################################################
  SEED: Radiohead — Kid A
######################################################################

Top 3 recommendations for: radiohead — kid a

  #1: endochine — day two (score: 0.8478)
  #2: idiot pilot — strange we should meet here (score: 0.846)
  #3: muse — showbiz (score: 0.8401)

Running explainer on top recommendations...

  radiohead — kid a (critique_brainz)
  → endochine — day two (pitchfork)
1. **Incorporation of non-traditional rock instrumentation**
   - Review A: "They conducted a brass ensemble ("The National Anthem"). They tried to transmute their previous dread into art"
   - Review B: "a shuffling, piano-dirge stew (complete with mellotron choir)"

2. **Emotional expression of modern anxiety and distress**
   - Review A: "it left all listeners with a sense of the madness of modern life"
   - Review B: "a TV with the message 'I hate you' repeated on its screen"

  radiohead — kid a (critique_brainz

In [6]:
albums_to_demo_2 = [
    ("jay-z", "the blueprint"),
    ("drake", "take care"),
    ("arcade fire", "funeral"),
    ("childish gambino", "because the internet"),
    ("vampire weekend", "vampire weekend"),
    ("daft punk", "random access memories"),
    ("amy winehouse", "back to black"),
    ("d'angelo", "voodoo"),
    ("robyn", "body talk"),
]

for artist, album in albums_to_demo_2:
    print(f"\n{'#'*70}")
    print(f"  SEED: {artist.title()} — {album.title()}")
    print(f"{'#'*70}")
    recommend_and_explain(artist, album, top_k=3)


######################################################################
  SEED: Jay-Z — The Blueprint
######################################################################

Top 3 recommendations for: jay-z — the blueprint

  #1: jay‐z — the hits collection volume one (score: 0.856)
  #2: jay‐z — the blueprint 3 (score: 0.8483)
  #3: jay‐z — american gangster (score: 0.8478)

Running explainer on top recommendations...

  jay-z — the blueprint (pitchfork)
  → jay‐z — the hits collection volume one (critique_brainz)
1. **Masterful production and sample selection as core strength**
   - Review A: "Retro soul samples are dull white, picked clean of lint and sanitized. They're wielded like pieces of a glitch track around Jay's words, coming in at all the right moments"
   - Review B: "Jay-Z's well-tuned ear for a crafty steal - a sample from the musical Annie, sitting high in the rap charts? Nobody saw that coming"

2. **Strategic collaborations with elite producers**
   - Review A: "rumor

In [7]:
albums_to_demo_3 = [
    ("usher", "confessions"),
    ("frank ocean", "blonde"),
]

for artist, album in albums_to_demo_3:
    print(f"\n{'#'*70}")
    print(f"  SEED: {artist.title()} — {album.title()}")
    print(f"{'#'*70}")
    recommend_and_explain(artist, album, top_k=3)


######################################################################
  SEED: Usher — Confessions
######################################################################

Top 3 recommendations for: usher — confessions

  #1: jacquees — king of r&b (score: 0.8279)
  #2: jacquees — 4275 (score: 0.8099)
  #3: r. kelly — untitled (score: 0.8079)

Running explainer on top recommendations...

  usher — confessions (critique_brainz)
  → jacquees — king of r&b (pitchfork)
1. **Autobiographical songwriting rooted in personal relationships**
   - Review A: "As the title implies, this is an autobiographical album, and the lyrics see him glide from the sexually charged (That's What It's Made For and Can U Handle It?) to the almost embarassingly personal."
   - Review B: "King of R&B, an hour-long, 18-track practice of the genre's techniques, is Jacquees' job application."

2. **Demonstration of technical mastery within R&B conventions**
   - Review A: "he's matured from a teen pop star to a sultr

In [8]:
recommend_and_explain("drake", "take care", top_k=10)


Top 10 recommendations for: drake — take care

  #1: stwo — d.t.s.n.t. (score: 0.8523)
  #2: majid jordan — majid jordan (score: 0.8494)
  #3: partynextdoor — partynextdoor (score: 0.8463)
  #4: tory lanez — i told you (score: 0.8436)
  #5: popcaan — vanquish (score: 0.8429)
  #6: wale — shine (score: 0.832)
  #7: tyga — careless world: rise of the last king (score: 0.8312)
  #8: tyga — the gold album: 18th dynasty (score: 0.8289)
  #9: partynextdoor — partymobile (score: 0.8261)
  #10: the foreign exchange — leave it all behind (score: 0.8257)

Running explainer on top recommendations...

  drake — take care (critique_brainz)
  → stwo — d.t.s.n.t. (pitchfork)
1. **Production aesthetic of muted, atmospheric beats designed for vocal performance**
   - Review A: "its muted backdrops - like listening to a record on your neighbour's stereo, through the dividing wall - were perfect for emoting over"
   - Review B: "The drums are programmed and flat, the mix is smoky and these beautiful voi

In [18]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

def llm_rerank(query_artist, query_album, top_k=10):
    artist_norm = query_artist.lower().strip()
    album_norm  = query_album.lower().strip()

    recs = all_recs_df[
        (all_recs_df["query_artist"] == artist_norm) &
        (all_recs_df["query_album"]  == album_norm)
    ].sort_values("rank").head(top_k)

    if recs.empty:
        print(f"Not found: {query_artist} — {query_album}")
        return

    print(f"Scoring {len(recs)} candidates with LLM...\n")

    results = []
    for _, row in recs.iterrows():
        rec_artist = row["rec_artist"]
        rec_album  = row["rec_album"]

        q_mask = (
            (df["album_norm"] == album_norm) &
            (df["artist_norm"] == artist_norm)
        )
        r_mask = (
            (df["album_norm"] == rec_album.lower().strip()) &
            (df["artist_norm"] == rec_artist.lower().strip())
        )

        q_reviews = df[q_mask]
        r_reviews = df[r_mask]

        if q_reviews.empty or r_reviews.empty:
            print(f"  Skipping {rec_artist} — {rec_album}: review not found")
            continue

        q_text   = q_reviews.iloc[0]["cleaned_text"]
        r_text   = r_reviews.iloc[0]["cleaned_text"]
        q_source = q_reviews.iloc[0]["dataset"]
        r_source = r_reviews.iloc[0]["dataset"]
        q_label  = f"{query_artist} — {query_album} ({q_source})"
        r_label  = f"{rec_artist} — {rec_album} ({r_source})"

        prompt = f"""You are analyzing two music reviews to find their most meaningful shared qualities.

Identify up to 3 significant qualities that both albums genuinely share, based ONLY on what is actually written in the reviews.
If fewer than 3 genuine connections exist, list only those that are real — do not pad with weak or generic observations.
If there are no meaningful connections, return exactly: NONE

For each shared quality:
- State it in one clear phrase
- Quote the specific language from each review that supports it (verbatim, in quotation marks, attributed to Review A or Review B)

Album A: {q_label}
Review A:
{q_text}

Album B: {r_label}
Review B:
{r_text}

Return a numbered list only. No introduction, no conclusion. If no genuine connections, return: NONE"""

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=400,
            messages=[{"role": "user", "content": prompt}]
        )
        result = response.content[0].text.strip()

        if result == "NONE":
            n_connections = 0
        else:
            n_connections = len([l for l in result.split("\n")
                                 if l.strip().startswith(("1.", "2.", "3."))])

        print(f"  #{row['rank']} {rec_artist} — {rec_album}: {n_connections} connections")

        results.append({
            "original_rank":  int(row["rank"]),
            "artist":         rec_artist,
            "album":          rec_album,
            "cosine_score":   row["score"],
            "n_connections":  n_connections,
            "explanation":    result
        })

    # Re-rank: first by n_connections (desc), then by cosine_score (desc)
    results.sort(key=lambda x: (x["n_connections"], x["cosine_score"]), reverse=True)

    print(f"\n{'Orig':>4}  {'New':>4}  {'Connections':>11}  {'Score':>6}  Artist — Album")
    print("-" * 70)
    for new_rank, r in enumerate(results, 1):
        print(f"{r['original_rank']:>4}  {new_rank:>4}  {r['n_connections']:>11}  "
              f"{r['cosine_score']:>6.4f}  {r['artist']} — {r['album']}")

    print(f"\nFull explanations for top 3 LLM-ranked results:\n")
    for r in results[:3]:
        print(f"\n{'='*60}")
        print(f"  {query_artist} — {query_album}")
        print(f"  → {r['artist']} — {r['album']}")
        print(f"  Cosine rank: #{r['original_rank']} | Connections: {r['n_connections']}")
        print(f"{'='*60}")
        print(r["explanation"])

    return results

llm_rerank("drake", "take care", top_k=10)

Scoring 10 candidates with LLM...

  #1 stwo — d.t.s.n.t.: 3 connections
  #2 majid jordan — majid jordan: 2 connections
  #3 partynextdoor — partynextdoor: 3 connections
  #4 tory lanez — i told you: 3 connections
  #5 popcaan — vanquish: 3 connections
  #6 wale — shine: 2 connections
  #7 tyga — careless world: rise of the last king: 3 connections
  #8 tyga — the gold album: 18th dynasty: 2 connections
  #9 partynextdoor — partymobile: 3 connections
  #10 the foreign exchange — leave it all behind: 3 connections

Orig   New  Connections   Score  Artist — Album
----------------------------------------------------------------------
   1     1            3  0.8523  stwo — d.t.s.n.t.
   3     2            3  0.8463  partynextdoor — partynextdoor
   4     3            3  0.8436  tory lanez — i told you
   5     4            3  0.8429  popcaan — vanquish
   7     5            3  0.8312  tyga — careless world: rise of the last king
   9     6            3  0.8261  partynextdoor — partymobil

[{'original_rank': 1,
  'artist': 'stwo',
  'album': 'd.t.s.n.t.',
  'cosine_score': 0.8523,
  'n_connections': 3,
  'explanation': '1. **Sparse, ambient production aesthetic** — Review A: "its muted backdrops - like listening to a record on your neighbour\'s stereo, through the dividing wall - were perfect for emoting over" and "the sparse, ambient arrangements typically favoured by Shebib"; Review B: "The drums are programmed and flat, the mix is smoky and these beautiful voices float atop it all, creating a drugged-out haze"\n\n2. **Prioritizing vocalists and emotional expression over producer signature** — Review A: "by surrounding himself with the right producers and guests...he\'s elevated himself to remarkable heights"; Review B: "His immaculately produced soundscapes do a great service to his vocalists...But he\'s lowered his own voice to a whisper"\n\n3. **Noah "40" Shebib\'s influential production blueprint** — Review A: "Calling predominantly on the production skills of Noah

In [19]:
albums_to_rerank = [
    ("radiohead", "kid a"),
    ("charli xcx", "pop 2"),
    ("janelle monáe", "the archandroid"),
    ("frank ocean", "channel orange"),
    ("taylor swift", "folklore"),
    ("kendrick lamar", "to pimp a butterfly"),
    ("sufjan stevens", "illinois"),
    ("bon iver", "for emma, forever ago"),
    ("beyoncé", "lemonade"),
    ("outkast", "speakerboxxx/the love below"),
]

for artist, album in albums_to_rerank:
    print(f"\n{'#'*70}")
    print(f"  SEED: {artist.title()} — {album.title()}")
    print(f"{'#'*70}")
    llm_rerank(artist, album, top_k=10)


######################################################################
  SEED: Radiohead — Kid A
######################################################################
Scoring 10 candidates with LLM...

  #1 endochine — day two: 2 connections
  #2 idiot pilot — strange we should meet here: 2 connections
  #3 muse — showbiz: 3 connections
  #4 aereogramme — sleep and release: 3 connections
  #5 thom yorke — the eraser: 3 connections
  #6 atoms for peace — amok: 3 connections
  #7 for stars — ...it falls apart: 3 connections
  #8 idlewild — the remote part: 2 connections
  #9 elbow — build a rocket boys!: 2 connections
  #10 cluster — qua: 3 connections

Orig   New  Connections   Score  Artist — Album
----------------------------------------------------------------------
   3     1            3  0.8401  muse — showbiz
   4     2            3  0.8386  aereogramme — sleep and release
   5     3            3  0.8381  thom yorke — the eraser
   6     4            3  0.8362  atoms for peace 